# Sesión 1 — Soluciones: Prompt Engineering

> **Nota para el instructor**: Este notebook contiene una solución de referencia para cada ejercicio. Las soluciones no son únicas — hay muchas formas correctas de formular un prompt. Lo importante es que los estudiantes entiendan el razonamiento detrás de cada elección de diseño.

## Setup

In [ ]:
%pip install -q google-genai pandas

In [ ]:
from google import genai
import pandas as pd
import json

from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

MODEL = 'gemini-2.5-flash'

print('✓ API configurada')

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/ber2/isdi-gen-ai/refs/heads/main/sesion1/amazon_electronics_50.csv'

try:
    sample = pd.read_csv(DATA_URL)
    print('✓ Dataset cargado desde GitHub')
except Exception:
    sample = pd.read_csv('sesion1/data/amazon_electronics_50.csv')
    print('✓ Dataset cargado desde fichero local')

print(f'Muestra lista: {len(sample)} reviews')

## Ejercicio 1 — Solución: Clasificación Básica

**Clave del diseño**: La instrucción `Responde SOLO con una palabra` es lo que evita que el modelo añada texto explicativo no deseado. Sin esta restricción, el modelo tiende a elaborar aunque no se lo pidamos.

In [ ]:
review = sample.iloc[0]
print(f'Rating real: {review["rating"]} ★')
print(f'Texto:\n{review["text"]}')

In [ ]:
prompt = f"""Clasifica la siguiente review de producto como exactamente una de estas tres opciones:
positiva, negativa o mixta.

Responde SOLO con una palabra. Sin explicaciones.

Review:
{review['text']}
"""

respuesta = client.models.generate_content(model=MODEL, contents=prompt)
print('Clasificación:', respuesta.text.strip())
print(f'(Rating real: {review["rating"]}★ — ¿coincide?)')

## Ejercicio 2 — Solución: Extracción de Información Estructurada

**Clave del diseño**: 
- Especificar el schema exacto del JSON con nombres de campo concretos
- La instrucción `Sin markdown, sin bloques de código` evita que el modelo envuelva el JSON en ` ```json ``` `
- Sin esa instrucción, `json.loads()` falla porque el texto contiene backticks

In [ ]:
review2 = sample.iloc[5]
print(f'Rating real: {review2["rating"]} ★')
print(f'Texto:\n{review2["text"]}')

In [ ]:
prompt2 = f"""Analiza la siguiente review de un producto electrónico y devuelve un JSON con exactamente estos campos:

{{
  "clasificacion": "positiva" | "negativa" | "mixta",
  "producto_mencionado": "nombre o tipo de producto inferido del texto",
  "aspectos_positivos": ["lista", "de", "aspectos", "valorados"],
  "aspectos_negativos": ["lista", "de", "quejas", "o", "problemas"]
}}

Responde SOLO con el JSON. Sin markdown. Sin bloques de código. Sin texto adicional.

Review:
{review2['text']}
"""

respuesta2 = client.models.generate_content(model=MODEL, contents=prompt2)
print('Respuesta raw:')
print(respuesta2.text)

print('\n--- Parseado como JSON ---')
try:
    datos = json.loads(respuesta2.text)
    for campo, valor in datos.items():
        print(f'{campo}: {valor}')
except json.JSONDecodeError as e:
    print(f'Error al parsear JSON: {e}')
    print('El modelo añadió texto extra. Ajustar el prompt.')

## Ejercicio 3 — Solución: Mejora del Prompt

**Qué diferencia la versión mejorada**:
1. **Rol específico**: convierte al modelo en un agente con perspectiva concreta
2. **Restricciones explícitas**: evita respuestas genéricas o defensivas
3. **Estructura pedida**: el modelo sabe exactamente qué debe incluir
4. **Límite de extensión**: evita respuestas largas e imprecisas

In [ ]:
reviews_negativas = sample[sample['rating'] <= 2]
review_negativa = reviews_negativas.iloc[0] if len(reviews_negativas) > 0 else sample.iloc[10]

print(f'Rating: {review_negativa["rating"]} ★')
print(f'Texto:\n{review_negativa["text"]}')

In [ ]:
# Versión básica — sin contexto ni restricciones
prompt_basico = f"""Responde a esta review negativa:

{review_negativa['text']}
"""

respuesta_basica = client.models.generate_content(model=MODEL, contents=prompt_basico)
print('=== VERSIÓN BÁSICA ===')
print(respuesta_basica.text)

In [ ]:
# Versión mejorada — con rol, tono, estructura y restricciones
prompt_mejorado = f"""Eres el responsable de atención al cliente de una tienda de electrónica online.
Tu tono es profesional, empático y orientado a soluciones — nunca defensivo ni genérico.

Un cliente ha publicado esta review de {review_negativa['rating']} estrella/s:
\"\"\"
{review_negativa['text']}
\"\"\"

Redacta una respuesta pública de exactamente 3 frases en español que:
1. Agradezca el feedback de forma sincera (no genérica)
2. Reconozca el problema específico mencionado
3. Proponga una acción concreta o invite a contactar para resolver el problema

NO uses frases como "Lamentamos los inconvenientes" o "Sentimos mucho que...".
NO prometas cosas que no puedes garantizar.
"""

respuesta_mejorada = client.models.generate_content(model=MODEL, contents=prompt_mejorado)
print('=== VERSIÓN MEJORADA ===')
print(respuesta_mejorada.text)

## Ejercicio 4 — Solución: Análisis Comparativo

**Clave del diseño**: Formatear claramente las reviews con separadores y número de etiqueta evita que el modelo mezcle los textos. La pregunta sobre contradicciones es importante para que el modelo no limite su análisis a encontrar consenso.

In [ ]:
tres_reviews = sample.iloc[[0, 1, 2]]

for i, (_, r) in enumerate(tres_reviews.iterrows(), 1):
    print(f'--- Review {i} (Rating: {r["rating"]}★) ---')
    print(r['text'][:300] + '...' if len(r['text']) > 300 else r['text'])
    print()

In [ ]:
textos = [r['text'] for _, r in tres_reviews.iterrows()]
ratings = [r['rating'] for _, r in tres_reviews.iterrows()]

reviews_formateadas = '\n\n'.join([
    f'--- Review {i+1} (Rating: {ratings[i]}★) ---\n{texto}'
    for i, texto in enumerate(textos)
])

prompt_comparativo = f"""Eres un analista de producto. Tienes 3 reviews de clientes sobre productos electrónicos.

{reviews_formateadas}

Realiza un análisis comparativo que responda:

1. **Puntos en común**: ¿Qué aspectos menciona más de un cliente? (positivos o negativos)
2. **Contradicciones**: ¿Hay opiniones opuestas sobre algún aspecto concreto?
3. **Recomendación**: Si alguien te preguntara si comprar un producto de este tipo basándote en estas reviews, ¿qué le dirías en 2 frases?

Responde en español. Sé concreto y directo.
"""

respuesta_comparativa = client.models.generate_content(model=MODEL, contents=prompt_comparativo)
print(respuesta_comparativa.text)